In [0]:

%sql
-- Creating Valume for raw data
CREATE VOLUME IF NOT EXISTS dbr_dev.artemzharkov10.raw_data;

In [0]:
import requests
import zipfile
import io
import os
import pyspark.sql.functions as F
from pyspark.sql.types import StructType,StructField,StringType,DoubleType,FloatType,IntegerType
from decimal import Decimal

In [0]:
# Is Executed importing the CSV Data of "Poland Houses Prices" from Kaggle via API KEY and Kaggle API
volume_path = "/Volumes/dbr_dev/artemzharkov10/raw_data"
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/dawidcegielski/house-prices-in-poland"
kaggle_token = "KGAT_a0356ab89cb4776324bbedf2b57ce23a"
# (CHANGEDKEYFOR_)GAT_(_PUSHON_)a0356ab89cb4776324bbedf2b57ce23a(_GIHAB)
print("Connect to Kaggle REST API")
headers = {"Authorization": f"Bearer {kaggle_token}"}
response = requests.get(dataset_url, headers=headers, stream=True)

if response.status_code == 200:
    print("Data is downloaded")
    print("Started unziping")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        for file_name in z.namelist():
            target_path = os.path.join(volume_path, file_name)
            with open(target_path, "wb") as f:
                f.write(z.read(file_name))
            print(f"File: {file_name} copleated")
            
    print(f"Done!")
else:
    print(f"Download error: {response.status_code}")
    print(response.text)



In [0]:
# Is Executed a transformation RAW_DATA to Bronze Table
file_path = "/Volumes/dbr_dev/artemzharkov10/raw_data/Houses.csv"
df = spark.read.csv(file_path, header=True,encoding="windows-1250")
df.write.format("delta").mode("overwrite").saveAsTable("dbr_dev.artemzharkov10.PolHousePrices_Bronze")   

In [0]:

# Transformation to Silver layer
file_path = "dbr_dev.artemzharkov10.PolHousePrices_Bronze"
bronze_df = spark.read.table(file_path).na.replace({"": None, "N/A": None})
# display(bronze_df.limit(10))
silver_df = (
    bronze_df.dropDuplicates()
    .withColumnRenamed("id", "house_id")
    .withColumnRenamed("_c0", "id")
    .withColumn("house_id", F.col("house_id").cast("double").cast("integer"))
    .withColumn("id", F.col("id").cast("integer"))
    .withColumn("address",F.initcap(F.lower(F.trim(F.col("address")))))
    .withColumn("city", F.initcap(F.lower(F.trim(F.col("city")))))
    .withColumn("floor",F.col("floor").cast("double").cast("integer"))
    .withColumn("latitude",F.round(F.col("latitude").cast("double"),7))
    .withColumn("longitude",F.round(F.col("longitude").cast("double"),7))
    .withColumn("price",F.round(F.col("price").cast("double"),2))
    .withColumn("currency_type", F.lit("PLN"))
    .withColumn("rooms",F.col("rooms").cast("double").cast("integer"))
    .withColumn("sq",F.round(F.col("sq").cast("double"),2).cast("double"))
    .withColumn("year",F.col("year").cast("double").cast("integer"))
    .withColumn("year", F.year(F.try_to_date(F.col("year").cast("string"), "yyyy")))
)
# display(silver_df.limit(10))

# Is Executed a transformation Bronze Table to Silver Table
# option("overwriteSchema", "true") 
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbr_dev.artemzharkov10.PolHousePrices_Silver")

In [0]:
#  OPEN Currency API via NBP (national bank of Poland)
# http://api.nbp.pl/api/exchangerates/rates/a/eur/?format=json
url = "https://api.nbp.pl/api/exchangerates/tables/a/?format=json"
data = None
try:
    print("Req to NBP...")
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        print("Data is obtained!")
except Exception as e:
    print(f"Error: {e}")
# print(data)

currency_list = []
if data is not None:
    try:
        for item in data:
            for rate in item["rates"]:
                currency_list.append({"code": rate["code"], "value": float(rate["mid"])})
    except Exception as e:
        print(f"Error: {e}")
# print(currency_list)
                     
    




In [0]:
# Was transformed SilverTable to Long Format (but how to avoid Shuffling/OOM with cross join ?)
# F.broadcast()

silver_df = spark.read.table("dbr_dev.artemzharkov10.PolHousePrices_Silver")

target_currency = ["USD","EUR","GBP"]
target_city= ["Kraków","Warszawa","Poznań"]
min_max_price = [500000,2000000]
min_max_sq = [30,400]

code_carrency_set = []
for item in currency_list:
    if item["code"] in target_currency:
        code_carrency_set.append((item["code"], item["value"]))
# print(code_carrency_set)

currency_schema = StructType([
    StructField("currency_type", StringType(), True),
    StructField("value", FloatType(), True)
])
currency_df = spark.createDataFrame(data=code_carrency_set, schema=currency_schema)

gold_df = (
    silver_df.alias("s")
        .filter(F.col("city").isin(target_city))
        .filter(F.col("price").between(min_max_price[0], min_max_price[1]))
        .filter(F.col("sq").between(min_max_sq[0], min_max_sq[1]))

        .withColumn("mean_sq_price_pln", F.round(F.col("price") / F.col("sq"), 2))
        # Have the same currency column name as the original DataFrame
        .crossJoin(F.broadcast(currency_df.alias("c")))
        .withColumn("converted_price", F.round(F.col("price") / F.col("value"), 2))
        .withColumn("converted_sq_price", F.round(F.col("mean_sq_price_pln") / F.col("value"), 2))

        .select("s.id",
                "s.city",
                F.col("s.price").alias("price_pln"),
                "mean_sq_price_pln",
                F.col("c.currency_type"),
                "converted_price",
                "converted_sq_price",
                "s.rooms",
                "s.sq",
                "s.year"
                )
        .orderBy("s.id", "c.currency_type")
      )
# display(df.limit(10))

gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dbr_dev.artemzharkov10.PolHousePrices_Gold")


In [0]:
%sql

-- I need materials about it
-- ??
OPTIMIZE dbr_dev.artemzharkov10.PolHousePrices_Gold 
-- ?? 
ZORDER BY (city, currency_type);
-- History log
DESCRIBE HISTORY dbr_dev.artemzharkov10.PolHousePrices_Gold;

In [0]:
%sql
SELECT city, currency_type, converted_sq_price AS sq_price
FROM dbr_dev.artemzharkov10.PolHousePrices_Gold

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT city, year, converted_sq_price AS sq_price
FROM dbr_dev.artemzharkov10.polhouseprices_gold
WHERE currency_type = 'USD' AND rooms < 4 AND year BETWEEN 1900 AND 2026 AND price_pln BETWEEN 10000 AND 1000000


Databricks visualization. Run in Databricks to view.